In [ ]:
1+1

## document loader

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader,PyPDFLoader
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException

logging=log



class DocumentLoader:
    def __init__(self,directory:str,):
        self.directory=directory
        logging.info("initialized docuemnt loader")

    def document_loader(self):
        rew_documents=PyPDFDirectoryLoader(self.directory)
        self.docs=rew_documents.load()
        if self.docs:
            try:
                pdf_files = set()
                for doc in self.docs:
                    source = doc.metadata.get('source', 'unknown')
                    pdf_files.add(source)
                # Log each PDF file
                logging.info(f"✅ Successfully loaded {len(self.docs)}  , documents lent is  {len(pdf_files)} PDF files:")
                for pdf in pdf_files:
                    logging.info(f" 📄 {pdf}")
                
            except Exception as e:
                    logging.error(f"error in loading documents : {e}")
                    raise CustomException(
                        message=f"Failed to load documents {e}",
                        error_detail=e
                    )
        elif not self.docs:
            logging.error(f" No documents found in {self.directory}")
            print(f"⚠️ No documents found in {self.directory}")
         
        return self.docs
    





In [ ]:
data = DocumentLoader("../data/")

In [ ]:
docs=data.document_loader()

In [ ]:
docs[0].metadata

In [ ]:
docs[0].metadata.get("source")

## embedings

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException
import sys

logging=log


class Embeddings:

    def __init__(self,
        model_name:str="sentence-transformers/all-MiniLM-L6-v2",
        device: str = "cpu",
        normalize_embeddings: bool = True,
        batch_size: int = 32):

        self.model_name = model_name
        self.device = device
        self.normalize_embeddings = normalize_embeddings
        self.batch_size = batch_size
        self._embeddings = None
        logging.info("embedding get initialized")


    def initializing_embedding(self):
        if self._embeddings is None:
            try:
                self.embeddings = HuggingFaceEmbeddings(
                        model_name=self.model_name,
                        model_kwargs={'device': self.device},
                        encode_kwargs={
                            'normalize_embeddings': self.normalize_embeddings,
                            'batch_size': self.batch_size
                        }
                    )
                log.info(f"initialiased embeding model {HuggingFaceEmbeddings.__class__.__name__} with model name : {self.model_name}")
            except Exception as e:
                    logging.error(f"error during initilizing embedings : {e}")
                    raise CustomException(
                        f"Failed to initialize embeddings with model {self.model_name} or their is an error in embeding models {e}",
                        sys
                    )
        return self.embeddings

## chunker

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker   # ← FIXED
from langchain_community.document_loaders import PyPDFDirectoryLoader
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException

logging=log



class TextSpliter:
    def __init__(self,embeding_model,persist_directory:str="vectorestore_VDB"):
        self.embeding=embeding_model
        self.persist_directory=persist_directory
        


    
    def split_documents(self,documents):
        # Split
        try:
            text_splitter = SemanticChunker(
                            embeddings=self.embeding,
                            breakpoint_threshold_type="percentile"
                        )

           
            chunks = text_splitter.split_documents(documents)
            logging.info(f"✅ Split {len(documents)} documents into {len(chunks)} chunks")
            return chunks
        except Exception as e:
            logging.error(f"❌ Error splitting documents: {e}")
            raise CustomException(f"Failed to chunk  {e}",sys)






## retriver

In [ ]:
from langchain_chroma import Chroma
from src.loging.logger import log
from src.exceptions.custom_exceptions import CustomException

logging=log
from pathlib import Path

class VectorStore:
    def __init__(self, embeddings, persist_directory):
        self.embeddings = embeddings
        self.persist_directory = persist_directory
        self.vectorstore = None
        Path(persist_directory).mkdir(parents=True, exist_ok=True)
        logging.info(f"✅ VectorStore initialized with persist_dir: {persist_directory}")

    def create_from_documents(self, documents):
        """Create vectorstore from documents"""
        try:
            self.vectorstore = Chroma.from_documents(
                documents=documents,
                embedding=self.embeddings,
                persist_directory=self.persist_directory
            )
            logging.info(f"✅ Created vectorstore with {len(documents)} documents")
            return self.vectorstore
        except Exception as e:
            logging.error(f"❌ Error creating vectorstore: {e}")
            raise CustomException("Failed to create vectorstore", e)

    def load_existing(self):
        """Load existing vectorstore"""
        try:
            self.vectorstore = Chroma(
                embedding_function=self.embeddings,
                persist_directory=self.persist_directory
            )
            logging.info(f"✅ Loaded existing vectorstore from {self.persist_directory}")
            print(f"✅ Loaded existing vectorstore from {self.persist_directory}")
            return self.vectorstore
        except Exception as e:
            logging.error(f"❌ Error loading vectorstore: {e}")
            raise CustomException("Failed to load vectorstore", e)

    def get_retriever(self, k: int = 4):
        """Get retriever from vectorstore"""
        if self.vectorstore is None:
            raise CustomException("Vectorstore not created yet. Call create_from_documents first.")
        return self.vectorstore.as_retriever(search_type="mmr",
                                             search_kwargs={"k": k,
                                                            "fetch_k": 20,
                                                            "lambda_mult": 0.5 })

In [ ]:
data = DocumentLoader("../data/PLAN_COMPTABLE")
docs=data.document_loader()
embeding_model=Embeddings()
embeding=embeding_model.initializing_embedding()
text_splietr=TextSpliter(embeding)
chunks=text_splietr.split_documents(docs)
vectore_store=VectorStore(embeding,"artifacts/vectorestore/CGI")
vectore_store.create_from_documents(chunks)
store=vectore_store.get_retriever()

In [ ]:
print(chunks[2].page_content)

In [ ]:
store.invoke("tva recuperable sur charge")

In [ ]:
from src.PipeLine.pipeline import  RagPipeLine

In [ ]:
pipeline=RagPipeLine(data_dir="../data/PLAN_COMPTABLE",persist_dir="artifacts/vectorestore/plan_comptable",force_rebuild=False)

In [ ]:
pip=pipeline.run()

In [ ]:
result=pip.invoke("les cadaux publicitaire dans  IR")
for i in range(len(result)):
    print(result[i].page_content)
    print("___________________________________________")

In [ ]:
result

In [ ]:
from src.tools.cgnc import cgnc_tool
from src.tools.finance_law import finance_law_tool

c:\dev\aaa\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-23 01:04:51,715 - INFO - embedding get initialized



🚀 RAG Pipeline initialized
📂 Data: ../data/CGNC
💾 Persist: artifacts/vectorestore/db_CGNC
🔍 Vectorstore exists: False



2026-05-23 01:04:52,262 - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-23 01:04:52,263 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-23 01:04:52,320 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-large/3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3/modules.json "HTTP/1.1 200 OK"
2026-05-23 01:04:52,510 - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-05-23 01:04:52,513 - INFO - Loading SentenceTransformer model from intfloat/multilingual-e5-large.
2026-05-23 01:04:52,674 - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-large/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Foun

In [ ]:
tools=[cgnc_tool,finance_law_tool]

In [ ]:
from  dotenv import  load_dotenv

from langchain_openrouter import ChatOpenRouter
import os

import os

load_dotenv()


os.environ["OPEN_ROUTER_API"] = os.getenv("OPEN_ROUTER_API")  

MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
llm = ChatOpenRouter(model=MODEL)

In [ ]:
llm.invoke("what is todays date")

In [ ]:
llm_with_tools=llm.bind_tools(tools)

In [ ]:
result=llm_with_tools.invoke("what is tva recuperable sur charge ")
print(result)

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage,SystemMessage

from langgraph.graph import StateGraph, START, END 
from  langgraph.prebuilt import ToolNode
from typing import TypedDict,List
from typing_extensions import Annotated
from langgraph.graph import add_messages

from  dotenv import  load_dotenv

from langchain_openrouter import ChatOpenRouter
import os

import os

os.environ["OPENROUTER_API_KEY"] = os.getenv("OPEN_ROUTER_API")  

MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
llm = ChatOpenRouter(model=MODEL)


llm_with_tools=llm.bind_tools(tools)



class AgentState(TypedDict):
    messages:Annotated[List,add_messages]

def chat_node(state: AgentState) -> dict:
    """A simple node that appends an AI response."""
    # `state["messages"]` already contains the full conversation history
    last_human_msg = state["messages"][-1].content if state["messages"] else ""

    response = llm_with_tools.invoke(last_human_msg)

    return {"messages": [response]}

tool_node=ToolNode(tools)

def agent_structuring_response(state: AgentState):
    # Find the last ToolMessage to know which tool was used
    
    system_message = SystemMessage(content=f"""You are a response structuring assistant.

                Instructions: generate the responce from the context u recive

                if you feel that you need to reuse the tools multiple time reuse multiple tools .
                
                If the response content is empty or unclear, say: "I couldn't find specific information on this topic. Try rephrasing your question."

                Do not add information that wasn't in the original response. Just structure and format it.
                """)

    all_messages = [system_message] + state["messages"][-5:]
    return {"messages": [llm_with_tools.invoke(all_messages)]}





# ── Build the graph ──
builder = StateGraph(AgentState)
builder.add_node("chat", chat_node)
builder.add_node("tool_node", tool_node)
builder.add_node("structures", agent_structuring_response)


builder.add_edge(START, "chat")
builder.add_edge("chat", "tool_node")
builder.add_edge("tool_node", "structures")

builder.add_edge("structures", END)



graph = builder.compile()



In [ ]:
# ── Invoke ──
result = graph.invoke({"messages": [HumanMessage(content="comment gerer les cadaux publicitaire")]})
result["messages"]

In [ ]:
import pprint

x=result["messages"][-1].content
pprint.pprint(x)